# Sesión 1 · LLMs y setup de entorno de trabajo

**Máster en AI Engineering · LIDR · Julián Fernández**

*Versión con los prompts en español.*

Hoy hacemos la llamada más básica de todas, y la miramos desde el punto de vista de producto:

| Bloque | Qué vemos |
|---|---|
| Configuración | Dependencias, claves y función de costes |
| 2 | Del prompt del usuario al prompt del producto |
| 3 | Parámetros del modelo y herramientas integradas |
| 4 | Comparativa de proveedores en vivo |
| 5 | Agregadores: una interfaz, varios proveedores |
| 6 | Tokens: cómo funcionan y cuánto cuestan |
| 7 | Arquitecturas reales y cierre |

Funciona igual en **Google Colab** (claves en el panel 🔑 *Secrets*) y en **local** (claves en un fichero `.env`).

## Configuración inicial

Tres celdas. Las tres tienen que terminar bien antes de pasar al Bloque 2.

In [1]:
# Config 1 · Dependencias
import importlib
import subprocess
import sys

try:
    import google.colab  # noqa: F401
    EN_COLAB = True
except ImportError:
    EN_COLAB = False

PAQUETES = ["openai", "anthropic", "tiktoken", "litellm"]

if EN_COLAB:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *PAQUETES])

import warnings
warnings.filterwarnings("ignore")  # avisos de dependencias de gradio/typer en Colab: inofensivos

for nombre in PAQUETES:
    modulo = importlib.import_module(nombre)
    version = getattr(modulo, "__version__", "")
    if not version:
        from importlib.metadata import version as _v
        version = _v(nombre)
    print(f"✅ {nombre} {version}")

✅ openai 3.51.0


✅ anthropic 2.8.0
✅ tiktoken 0.14.0


✅ litellm 1.109.1


In [2]:
# Config 2 · Claves y clientes
import os

def leer_clave(nombre: str) -> str | None:
    """Colab Secrets si estamos en Colab; si no, variables de entorno o el fichero .env."""
    if EN_COLAB:
        from google.colab import userdata
        try:
            return userdata.get(nombre)
        except Exception:
            return None
    try:
        from dotenv import load_dotenv
        load_dotenv(".env")
    except ImportError:
        pass
    return os.environ.get(nombre) or None

from openai import OpenAI
from anthropic import Anthropic

OPENAI_API_KEY = leer_clave("OPENAI_API_KEY")
ANTHROPIC_API_KEY = leer_clave("ANTHROPIC_API_KEY")

openai_client = OpenAI(api_key=OPENAI_API_KEY) if OPENAI_API_KEY else None
anthropic_client = Anthropic(api_key=ANTHROPIC_API_KEY) if ANTHROPIC_API_KEY else None

# LiteLLM lee las claves del entorno
if OPENAI_API_KEY:
    os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
if ANTHROPIC_API_KEY:
    os.environ["ANTHROPIC_API_KEY"] = ANTHROPIC_API_KEY

print("OpenAI:   ", "OK" if openai_client else "falta OPENAI_API_KEY")
print("Anthropic:", "OK" if anthropic_client else "falta ANTHROPIC_API_KEY (las celdas de Anthropic lo avisarán)")
if openai_client:
    print("Clientes configurados correctamente")

OpenAI:    OK
Anthropic: OK
Clientes configurados correctamente


In [3]:
# Config 3 · Función de costes
# Precios en dólares por millón de tokens (entrada, salida). Comprobados el 16/11/2026
# en las páginas oficiales de precios de OpenAI y Anthropic.
PRICES = {
    "gpt-4o-mini":       (0.15, 0.60),
    "gpt-4o":            (2.50, 10.00),
    "gpt-4.1-mini":      (0.40, 1.60),
    "gpt-5.4-nano":      (0.20, 1.25),
    "gpt-5.4-mini":      (0.75, 4.50),
    "gpt-5.4":           (2.50, 15.00),
    "gpt-5.5":           (5.00, 30.00),
    "claude-haiku-4-5":  (1.00, 5.00),
    "claude-sonnet-4-6": (3.00, 15.00),
}

def precio(modelo: str) -> tuple[float, float]:
    """Busca el precio por prefijo: 'claude-haiku-4-5-20251001' usa el de 'claude-haiku-4-5'."""
    modelo = modelo.split("/")[-1]  # 'openai/gpt-4o-mini' → 'gpt-4o-mini'
    for nombre in sorted(PRICES, key=len, reverse=True):
        if modelo.startswith(nombre):
            return PRICES[nombre]
    raise KeyError(f"Sin precio para {modelo}")

def calcular_coste(modelo: str, tokens_entrada: int, tokens_salida: int) -> float:
    p_in, p_out = precio(modelo)
    return (tokens_entrada * p_in + tokens_salida * p_out) / 1_000_000

def resumen_uso(modelo: str, tokens_entrada: int, tokens_salida: int) -> str:
    coste = calcular_coste(modelo, tokens_entrada, tokens_salida)
    return f"[{modelo}] entrada {tokens_entrada} · salida {tokens_salida} · ${coste:.6f}"

print("Función de cálculo de costes cargada")
print(resumen_uso("gpt-4o-mini", 1_000, 500))

Función de cálculo de costes cargada
[gpt-4o-mini] entrada 1000 · salida 500 · $0.000450


---
# Bloque 2 · Del prompt del usuario al prompt del producto

En un producto, la calidad de la respuesta **no puede depender de que el usuario sepa escribir prompts**.
La diseña quien desarrolla, y su herramienta principal es el *system prompt*.

## 2.1 · El problema: la respuesta genérica

In [4]:
# 2.1 · Sin system prompt
MODEL = "gpt-4o-mini"

respuesta = openai_client.responses.create(
    model=MODEL,
    input="Estamos migrando nuestro servicio de pagos de un monolito a microservicios. ¿Qué deberíamos tener en cuenta?",
)

print(respuesta.output_text)
print()
print(f"{len(respuesta.output_text.split())} palabras")
print(resumen_uso(MODEL, respuesta.usage.input_tokens, respuesta.usage.output_tokens))

Migrar un servicio de pagos de un monolito a una arquitectura de microservicios es un proceso complejo. Aquí hay varios aspectos importantes que debes tener en cuenta:

### 1. **Diseño de Microservicios**
   - **Descomposición del Monolito**: Identifica las diferentes funcionalidades dentro del monolito y agrúpalas lógicamente en microservicios. Por ejemplo, podrías tener servicios separados para la gestión de transacciones, la autenticación, el manejo de fraudes, etc.
   - **Límite de Contexto**: Define claramente los límites de cada microservicio para minimizar la dependencia entre ellos.

### 2. **Comunicación entre Microservicios**
   - **Protocolo de Comunicación**: Decide cómo se comunicarán los microservicios (REST, gRPC, mensajería asíncrona, etc.).
   - **Gestión de Errores y Retries**: Implementa estrategias para manejar errores y reintentos en las comunicaciones.

### 3. **Persistencia de Datos**
   - **Base de Datos por Servicio**: Considera si cada microservicio debería te

¿Pondríais esto en un producto? ¿Se lo daríais a un cliente que paga por vuestro software?

## 2.2 · El system prompt como contrato de calidad

La pregunta del usuario se fija **una vez**. Lo único que cambia en las cuatro iteraciones es el system prompt.

In [5]:
# 2.2 · La pregunta del usuario (no cambia)
USER_QUESTION = "Estamos migrando nuestro servicio de pagos de un monolito a microservicios. ¿Qué deberíamos tener en cuenta?"

def preguntar(system_prompt: str, pregunta: str = USER_QUESTION, model: str = MODEL, **params):
    """Una llamada a la Responses API: 'instructions' es el system prompt, 'input' lo que escribe el usuario."""
    r = openai_client.responses.create(model=model, instructions=system_prompt, input=pregunta, **params)
    print(r.output_text)
    print()
    print(f"{len(r.output_text.split())} palabras · {resumen_uso(model, r.usage.input_tokens, r.usage.output_tokens)}")
    return r

In [6]:
# 2.2 · Iteración 1 — rol básico
SYSTEM_PROMPT_V1 = "Eres un arquitecto de software sénior con experiencia en sistemas distribuidos."

r1 = preguntar(SYSTEM_PROMPT_V1)

Migrar un servicio de pagos de un monolito a microservicios es un proceso complejo que requiere una planificación cuidadosa y la consideración de varios aspectos clave. Aquí hay algunos puntos que deberías tener en cuenta:

### 1. **Análisis del Monolito Actual**
   - **Identificar Componentes**: Comprende la arquitectura actual del monolito, identificando los diferentes componentes y su interdependencia.
   - **Mapeo de Funcionalidades**: Define qué funcionalidades serán migradas a microservicios y cómo se agruparán.

### 2. **Definición de Microservicios**
   - **Responsabilidad Única**: Cada microservicio debe tener una responsabilidad claramente definida.
   - **Límites de Contexto**: Utiliza patrones como DDD (Domain-Driven Design) para definir límites de contexto y asegurar que los microservicios estén bien encapsulados.

### 3. **Comunicación entre Microservicios**
   - **Protocolos**: Decide si usarás comunicación síncrona (REST, gRPC) o asíncrona (message brokers, eventos).
  

In [7]:
# 2.2 · Iteración 2 — rol + audiencia + formato
SYSTEM_PROMPT_V2 = """Eres un arquitecto de software sénior con más de 15 años de experiencia construyendo infraestructura de pagos.

Tu audiencia: desarrolladores backend con más de 5 años de experiencia que nunca han liderado una migración como esta.

Formato: prosa corrida, sin listas de viñetas, como máximo 250 palabras. Cada párrafo debe contener una idea accionable."""

r2 = preguntar(SYSTEM_PROMPT_V2)

Al migrar de un monolito a microservicios, lo primero que debes considerar es la descomposición del servicio de pagos en componentes más pequeños y autónomos. Identifica las funcionalidades clave, como la gestión de transacciones, la conciliación de pagos y la gestión de usuarios, y planifica cómo cada microservicio interactuará con los demás. Esta descomposición no solo ayuda a la escalabilidad, sino que también facilita la implementación de cambios y mejoras sin afectar a todo el sistema.

Es fundamental pensar en la comunicación entre microservicios. La elección del protocolo, ya sea REST, gRPC o eventos asíncronos con un sistema de mensajería, influirá directamente en la latencia, la escalabilidad y la complejidad de la integración. Optar por un sistema basado en eventos puede ofrecer beneficios en términos de resiliencia y permitir una desacoplamiento más efectivo entre los servicios.

La seguridad también debe ser una prioridad. Implementa controles de acceso y autenticación robu

In [8]:
# 2.2 · Iteración 3 — restricciones de comportamiento
SYSTEM_PROMPT_V3 = SYSTEM_PROMPT_V2 + """

Comportamiento:
- Cuando menciones un compromiso (trade-off), indica su consecuencia concreta (latencia, coste, esfuerzo del equipo o riesgo).
- Si a la pregunta le falta contexto, explicita tus supuestos antes de responder.
- No recomiendes proveedores ni productos comerciales concretos.
- Menciona al menos un riesgo que no sea obvio."""

r3 = preguntar(SYSTEM_PROMPT_V3)

Al migrar de un monolito a una arquitectura de microservicios en un sistema de pagos, es crucial considerar la desacoplabilidad de los servicios. Cada microservicio debe ser autónomo y responsable de una función específica, lo que facilitará las actualizaciones y la escalabilidad. Sin embargo, este desacoplamiento viene con un coste en términos de latencia, ya que las interacciones interservicio podrían introducir retardos que no estaban presentes en el monolito, afectando la experiencia del usuario.

La gestión de transacciones es otro aspecto crítico. En un monolito, las transacciones son fáciles de manejar, pero en un sistema distribuido deberás implementar mecanismos de compensación o sagas para asegurar la consistencia. Esto aumenta el complejo desafío de gestionar el estado de la transacción, lo que puede generar mayor esfuerzo del equipo y un riesgo adicional de inconsistencias si no se implementa adecuadamente.

Además, la observabilidad es fundamental. A medida que introduces 

In [9]:
# 2.2 · Iteración 4 — contrato completo de producción
PRODUCTION_SYSTEM_PROMPT = """Eres un ingeniero principal que ha liderado la migración de una plataforma de pagos con certificación PCI-DSS,
que procesa más de 1.000 millones de dólares al año, de un monolito a microservicios.

Audiencia: desarrolladores backend con más de 5 años de experiencia que nunca han liderado una migración como esta.

Estructura de la respuesta (síguela al pie de la letra):
1. Empieza con la decisión más crítica que deben tomar primero, en una sola frase.
2. Después da 2-3 consideraciones. Para cada una: qué hacer, por qué y qué pasa si se equivocan.
3. Cierra con un riesgo que los equipos suelen pasar por alto.

Restricciones:
- Prosa corrida, como máximo 250 palabras.
- Si a la pregunta le falta contexto, indica tus supuestos en una frase.
- Cada compromiso (trade-off) debe incluir una consecuencia concreta.
- No recomiendes proveedores ni productos comerciales.
- No uses frases de relleno como «¡Buena pregunta!» ni «Depende» sin cuantificar."""

r4 = preguntar(PRODUCTION_SYSTEM_PROMPT)

La decisión más crítica que deben tomar primero es definir claramente los límites de los microservicios.

Primero, consideren la separación de dominios; identifiquen servicios específicos como procesamiento de pagos, gestión de usuarios y conciliación. Hacerlo asegura que cada microservicio tenga una responsabilidad clara, lo que reduce la complejidad. Si se equivocan y crean servicios demasiado amplios, enfrentarán problemas de acoplamiento, lo que perjudica la escalabilidad y la mantenibilidad.

Segundo, establezcan un enfoque robusto para la comunicación entre microservicios; evalúen si utilizarán REST, gRPC o colas. La elección impactará en la latencia y la fiabilidad. Si optan por una opción inadecuada, podrían enfrentar problemas de escalabilidad y rendimiento que afecten la experiencia del usuario final.

Finalmente, un riesgo que a menudo se pasa por alto es la gestión de la configuración y el control de versiones. Desatender este aspecto puede llevar a incompatibilidades en en

Tras cada iteración: **¿qué ha cambiado y por qué?** Cada línea del system prompt es una decisión de producto.

## 2.3 · Abstraer al usuario del prompting — versión ingenua frente a versión de producción

Un revisor de *pull requests*. El usuario solo pega un diff; nunca ve el prompt.

In [10]:
# 2.3 · El diff que revisamos (código de pagos con fallos)
DIFF = """
+def charge_customer(customer_id, amount):
+    customer = db.query(f"SELECT * FROM customers WHERE id = {customer_id}")
+    total = amount * 1.21
+    result = stripe.Charge.create(amount=total, currency="eur", customer=customer["stripe_id"])
+    db.execute(f"UPDATE orders SET paid = 1 WHERE customer_id = {customer_id}")
+    return result
"""
print(f"El usuario escribe {len(DIFF.strip().splitlines())} líneas de código.")

El usuario escribe 6 líneas de código.


In [11]:
# 2.3 · review_pull_request_naive()
def review_pull_request_naive(diff: str) -> str:
    r = openai_client.responses.create(
        model=MODEL,
        instructions="Eres un revisor de código.",
        input=diff,
    )
    print(resumen_uso(MODEL, r.usage.input_tokens, r.usage.output_tokens))
    return r.output_text

print(review_pull_request_naive(DIFF))

[gpt-4o-mini] entrada 108 · salida 521 · $0.000329
Aquí tienes una revisión del código:

1. **Inyección SQL**: La consulta SQL está vulnerable a inyección de SQL. Considera usar parámetros en lugar de interpolar el `customer_id` directamente en la consulta.

2. **Manejo de errores**: No hay manejo de excepciones. Si `stripe.Charge.create` falla, o si la consulta SQL no devuelve un resultado, se generará una excepción. Sería bueno envolver el código en un bloque `try-except`.

3. **Verificación del cliente**: Debes comprobar si el cliente se encontró antes de intentar acceder a `customer["stripe_id"]`. Si no se encuentra, deberías manejar ese caso adecuadamente.

4. **Redondeo de montos**: Asegúrate de que el monto total esté correctamente redondeado antes de enviarlo a Stripe. Stripe espera el monto en centavos y enteros.

5. **Uso de constantes**: Si `1.21` tiene un significado particular (por ejemplo, el IVA), considera usar una constante con un nombre descriptivo.

6. **Retorno del 

In [12]:
# 2.3 · review_pull_request() — prompt de producción
PR_REVIEW_SYSTEM_PROMPT = """Eres un ingeniero backend sénior que revisa pull requests para un equipo de pagos.
El código de pagos maneja dinero real y datos personales, así que la corrección y la seguridad van primero.

Criterios de revisión, en estricto orden de prioridad:
1. Seguridad: inyección, secretos, tratamiento inseguro de datos personales o de tarjetas.
2. Exactitud con el dinero: redondeo, unidades (céntimos frente a euros), divisa, cobros duplicados, idempotencia.
3. Consistencia de los datos: qué pasa si un paso falla a mitad (transacciones, reintentos).
4. Legibilidad y tests, solo si los puntos anteriores están limpios.

Formato de salida (Markdown):
- Primera línea: un veredicto, exactamente uno de APROBAR, PEDIR CAMBIOS o BLOQUEAR.
- Después, una lista numerada de hallazgos, del más grave al menos grave. Cada hallazgo: gravedad (crítica/mayor/menor),
  la línea exacta, qué está mal y un fragmento corregido.
- Como máximo 5 hallazgos. No incluyas detalles de estilo si hay algún hallazgo crítico.

Comportamiento:
- Sé directo y concreto. Sin elogios ni relleno.
- Si no estás seguro de que algo sea un bug, di qué tendrías que comprobar.
- Nunca inventes contexto que no esté en el diff."""

def review_pull_request(diff: str, language: str = "Python", service: str = "pagos") -> str:
    # Lo que el usuario aporta son sus datos; el prompt lo construye el producto.
    user_input = f"Lenguaje: {language}\nServicio: {service}\n\nDiff a revisar:\n```diff\n{diff}\n```"
    r = openai_client.responses.create(
        model=MODEL,
        instructions=PR_REVIEW_SYSTEM_PROMPT,
        input=user_input,
        temperature=0.2,
    )
    print(resumen_uso(MODEL, r.usage.input_tokens, r.usage.output_tokens))
    return r.output_text

print(f"System prompt: {len(PR_REVIEW_SYSTEM_PROMPT.split())} palabras que el usuario nunca ve.\n")
print(review_pull_request(DIFF))

System prompt: 189 palabras que el usuario nunca ve.



[gpt-4o-mini] entrada 417 · salida 454 · $0.000335
BLOQUEAR

1. **Gravedad: crítica**  
   **Línea exacta:** `customer = db.query(f"SELECT * FROM customers WHERE id = {customer_id}")`  
   **Qué está mal:** Vulnerabilidad de inyección SQL al usar interpolación de cadenas.  
   **Fragmento corregido:** `customer = db.query("SELECT * FROM customers WHERE id = %s", (customer_id,))`

2. **Gravedad: mayor**  
   **Línea exacta:** `total = amount * 1.21`  
   **Qué está mal:** El cálculo de `total` no considera la unidad correcta (céntimos vs. euros) y puede llevar a cobros incorrectos.  
   **Fragmento corregido:** `total = int(amount * 1.21 * 100)`  # Asumiendo que `amount` está en euros.

3. **Gravedad: mayor**  
   **Línea exacta:** `result = stripe.Charge.create(amount=total, currency="eur", customer=customer["stripe_id"])`  
   **Qué está mal:** Falta de manejo de errores en la llamada a la API de Stripe, lo que puede causar fallos silenciosos.  
   **Fragmento corregido:** 
   ```pyth

### Tu turno (opcional)

Piensa en una tarea de tu trabajo y escribe su función: el usuario aporta los datos y tú el prompt.

In [13]:
# 2.3 · Tu turno
MI_SYSTEM_PROMPT = """Eres ...

Audiencia: ...

Formato: ...

Restricciones:
- ..."""

def mi_funcion(datos_del_usuario: str) -> str:
    r = openai_client.responses.create(model=MODEL, instructions=MI_SYSTEM_PROMPT, input=datos_del_usuario)
    return r.output_text

# print(mi_funcion("..."))

## 2.4 · Consistencia de respuestas

¿Es aceptable que dos usuarios reciban respuestas de calidad diferente ante la misma pregunta?

In [14]:
# 2.4 · temperature 1.0 + prompt vago → cuatro respuestas distintas
for i in range(4):
    r = openai_client.responses.create(
        model=MODEL,
        instructions="Ayuda al usuario.",
        input="¿Cuál es el principal riesgo de migrar los pagos a microservicios? Responde en una o dos frases.",
        temperature=1.0,
        max_output_tokens=120,
    )
    print(f"--- {i + 1} ---\n{r.output_text}\n")

--- 1 ---
El principal riesgo de migrar los pagos a microservicios es la complejidad en la gestión de las interacciones entre servicios, lo que puede aumentar la probabilidad de errores y vulnerabilidades de seguridad si no se implementan adecuadamente. Además, puede haber desafíos en la garantía de la consistencia y la disponibilidad de los datos a través de múltiples servicios.



--- 2 ---
El principal riesgo de migrar los pagos a microservicios es la complejidad en la gestión y coordinación de múltiples servicios, lo que puede llevar a vulnerabilidades de seguridad y dificultades en la transacción de datos en tiempo real. Además, una mala implementación puede causar interrupciones en el servicio y afectar la experiencia del usuario.



--- 3 ---
El principal riesgo de migrar los pagos a microservicios es la complejidad en la gestión de las transacciones y la coordinación entre servicios, lo que puede aumentar la vulnerabilidad a fallos y afectar la seguridad y la integridad de los datos financieros.



--- 4 ---
El principal riesgo de migrar los pagos a microservicios es la complejidad en la gestión de la comunicación entre servicios, lo que puede llevar a problemas de seguridad, consistencia de los datos y dificultades en el seguimiento de transacciones. Además, una arquitectura microservicios mal implementada puede resultar en mayor latencia y costos operativos elevados.



In [15]:
# 2.4 · temperature 0.1 + prompt preciso → cuatro respuestas casi idénticas
PRECISE_PROMPT = """Responde exactamente con este formato y nada más:
Riesgo: <un sintagma nominal, como máximo 6 palabras>
Por qué: <una frase, como máximo 20 palabras>"""

for i in range(4):
    r = openai_client.responses.create(
        model=MODEL,
        instructions=PRECISE_PROMPT,
        input="¿Cuál es el principal riesgo de migrar los pagos a microservicios?",
        temperature=0.1,
        max_output_tokens=80,
    )
    print(f"--- {i + 1} ---\n{r.output_text}\n")

--- 1 ---
Riesgo: Complejidad en la integración de sistemas  
Por qué: La migración puede generar problemas de comunicación entre microservicios y sistemas existentes.



--- 2 ---
Riesgo: Complejidad en la integración de sistemas  
Por qué: La migración puede generar dificultades en la comunicación entre microservicios y sistemas existentes.



--- 3 ---
Riesgo: Complejidad en la integración de sistemas  
Por qué: La migración puede generar dificultades en la comunicación entre microservicios y sistemas existentes.



--- 4 ---
Riesgo: Complejidad en la integración de sistemas  
Por qué: La migración puede generar problemas de comunicación entre microservicios y sistemas existentes.



Los modelos son estocásticos. La consistencia la ponen **tus instrucciones y tus parámetros**:
`temperature`, `max_output_tokens` y *structured outputs* (en la Sesión 4).

---
# Bloque 3 · Parámetros del modelo y herramientas integradas

## 3.1 · Temperature

In [16]:
# 3.1 · Misma pregunta, tres temperaturas
NAMING = "Sugiere un nombre para un SaaS que estima el coste de proyectos de software. Responde solo con el nombre."

for temperatura in (0.0, 0.7, 1.5):
    nombres = []
    for _ in range(3):
        r = openai_client.responses.create(model=MODEL, input=NAMING, temperature=temperatura, max_output_tokens=20)
        nombres.append(r.output_text.strip())
    print(f"temperature={temperatura}: {nombres}")

temperature=0.0: ['CosteSoft', 'CosteSoft', 'CosteSoft']


temperature=0.7: ['CostePro', 'CosteSoft', 'CosteSoft']


temperature=1.5: ['CosteWise', 'CosteWise', 'CosteaPro']


`top_p` es la alternativa a `temperature`: **usa uno u otro, nunca los dos**. En Claude 4.5 y posteriores la API
rechaza combinarlos.

En producción, empieza con `temperature` entre 0.2 y 0.3 y súbela solo si necesitas creatividad.

## 3.2 · max_output_tokens

In [17]:
# 3.2 · Tres techos de salida
for limite in (50, 200, 800):
    r = openai_client.responses.create(
        model=MODEL,
        instructions=PRODUCTION_SYSTEM_PROMPT,
        input=USER_QUESTION,
        max_output_tokens=limite,
    )
    motivo = r.incomplete_details.reason if r.incomplete_details else "-"
    print(f"max_output_tokens={limite:>4} → status={r.status:<10} motivo={motivo:<18} "
          f"salida={r.usage.output_tokens} tokens")

max_output_tokens=  50 → status=incomplete motivo=max_output_tokens  salida=50 tokens


max_output_tokens= 200 → status=incomplete motivo=max_output_tokens  salida=200 tokens


max_output_tokens= 800 → status=completed  motivo=-                  salida=269 tokens


`max_output_tokens` es un **techo de seguridad**, no un control de longitud: con 50 la respuesta se corta a medias
(`status="incomplete"`). La longitud real se pide en el system prompt.

En Anthropic (`max_tokens`) es **obligatorio**; en OpenAI y Gemini, opcional.

## 3.3 · stop y seed

Estos dos van por la interfaz **Chat Completions**, la otra API de OpenAI (la volveremos a ver con LiteLLM).

In [18]:
# 3.3 · stop: el modelo se para al generar la secuencia
r = openai_client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "Enumera las 6 fases de un proyecto de software, numeradas del 1. al 6."}],
    stop=["4."],
)
print(r.choices[0].message.content)
print("finish_reason:", r.choices[0].finish_reason)

Claro, aquí tienes las 6 fases típicas de un proyecto de software:

1. **Planificación**: Definición de los objetivos, alcance, recursos y cronograma del proyecto. Análisis de viabilidad y elaboración del plan de trabajo.

2. **Análisis de requisitos**: Recolección y análisis de los requisitos del sistema, tanto funcionales como no funcionales. Entrevistas con usuarios y partes interesadas para entender sus necesidades.

3. **Diseño**: Creación de la arquitectura del sistema y diseño detallado de componentes. Se definen las interfaces, bases de datos y la interacción entre los distintos módulos.


finish_reason: stop


In [19]:
# 3.3 · seed: misma semilla, misma respuesta (casi siempre)
salidas = []
for _ in range(3):
    r = openai_client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": "Inventa una historia de dos frases sobre una base de datos."}],
        temperature=1.0,
        seed=42,
    )
    salidas.append(r.choices[0].message.content)
    print(f"[{r.system_fingerprint}] {r.choices[0].message.content}\n")

print("¿Las tres iguales?", len(set(salidas)) == 1)

[fp_274019df43] La base de datos del pequeño pueblo guardaba secretos de generaciones, donde cada fila era un susurro de historias olvidadas. Una noche, un joven curioso descubrió una entrada misteriosa que revelaba el paradero de un tesoro escondido, cambiando su vida para siempre.



[fp_274019df43] En una lejana galaxia, una antigua base de datos guardaba los secretos de civilizaciones olvidadas, esperando pacientemente a que un viajero curioso descifrara sus enigmáticos códigos. Un día, un aventurero audaz la activó accidentalmente, desatando poderes que transformaron su realidad y la de todos los seres vivos a su alrededor.



[fp_274019df43] La base de datos del pequeño pueblo guardaba secretos de generaciones, donde cada fila era un susurro de historias olvidadas. Una noche, un joven curioso descubrió una entrada misteriosa que revelaba el paradero de un tesoro escondido, cambiando su vida para siempre.

¿Las tres iguales? False


`seed` es reproducibilidad **best effort**: si cambia el `system_fingerprint` (el backend del proveedor), puede
cambiar la respuesta.

## 3.4 · Resumen de parámetros

| Tarea | temperature | max_output_tokens | Otros |
|---|---|---|---|
| Extracción de datos (facturas, formularios) | 0 – 0.2 | justo para el JSON | *structured outputs* |
| Clasificación / etiquetado | 0 | pequeño | `stop` o etiquetas cerradas |
| Asistente técnico, soporte | 0.2 – 0.4 | 300 – 800 | system prompt con formato |
| Redacción, resúmenes | 0.5 – 0.7 | según el texto | |
| Brainstorming, naming | 0.8 – 1.2 | pequeño | varias muestras |

Son puntos de partida, no verdades. Los **modelos de razonamiento** (familia gpt-5 con razonamiento, o-series)
no aceptan la mayoría de estos parámetros: se controlan con `reasoning.effort`.

## 3.5 · Herramientas integradas

Hasta ahora el modelo solo sabe lo que aprendió en el entrenamiento: nada reciente, nada privado y no ejecuta
código. Las herramientas integradas las ejecuta **el proveedor en sus servidores**; el modelo decide cuándo
usarlas y todo ocurre **en una sola llamada**.

| Herramienta | OpenAI (Responses API) | Anthropic (Messages API) |
|---|---|---|
| Búsqueda web | `web_search` | `web_search_20250305` (habilitar en la Console) |
| Búsqueda en documentos | `file_search` (vector stores) | — (Files API + contexto) |
| Ejecutar código | `code_interpreter` | `code_execution` |
| Controlar un ordenador | `computer_use_preview` | `computer_use` |
| Servidores MCP remotos | `mcp` | conector MCP |

## 3.6 · web_search con OpenAI

In [20]:
# 3.6 · web_search (OpenAI)
DEVDAY = "¿Cuáles fueron los anuncios más importantes del OpenAI DevDay 2025? Tres viñetas."

r_web = openai_client.responses.create(
    model="gpt-4o",  # no todos los modelos soportan la herramienta: gpt-4o sí
    tools=[{"type": "web_search"}],
    input=DEVDAY,
)
print(r_web.output_text)
print()
print(resumen_uso("gpt-4o", r_web.usage.input_tokens, r_web.usage.output_tokens))
print("Pasos:", [item.type for item in r_web.output])

Aquí tienes tres de los anuncios más importantes del OpenAI DevDay 2025, presentados en viñetas:

• Apps en ChatGPT (Apps SDK en vista previa): los desarrolladores ahora pueden crear experiencias interactivas dentro de ChatGPT, permitiendo que aplicaciones de terceros —como Canva o Zillow— se integren directamente en una conversación ([openai.com](https://openai.com/devday/2025/?utm_source=openai)).

• AgentKit: una nueva suite de herramientas para construir, desplegar y gestionar agentes AI de grado de producción. Incluye funcionalidades como Agent Builder, conexiones con herramientas externas, guardrails (mecanismos de seguridad) y capacidades de evaluación ([openai.com](https://openai.com/devday/2025/?utm_source=openai)).

• Nuevos modelos disponibles en la API: GPT‑5 Pro (orientado a tareas de alta precisión), Sora 2 (modelos de generación de video), además de versiones más económicas como gpt‑realtime‑mini (voz) y gpt‑image‑1‑mini (imágenes) ([openai.com](https://openai.com/devday

In [21]:
# 3.6 · Las fuentes, en las annotations
fuentes = {}
for item in r_web.output:
    if item.type == "message":
        for parte in item.content:
            for a in getattr(parte, "annotations", None) or []:
                if a.type == "url_citation":
                    fuentes[a.url] = a.title

for url, titulo in fuentes.items():
    print(f"- {titulo}\n  {url}")

- DevDay 2025 | OpenAI
  https://openai.com/devday/2025/?utm_source=openai


Si tu producto busca en la web, **enseñar las fuentes no es opcional**: es UX y es confianza.

## 3.7 · web_search con Anthropic

Puede fallar si la organización no ha habilitado la búsqueda web en la Claude Console: es una decisión de
gobernanza del administrador.

In [22]:
# 3.7 · web_search (Anthropic)
try:
    if anthropic_client is None:
        raise RuntimeError("No hay ANTHROPIC_API_KEY configurada.")
    r_claude = anthropic_client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=800,  # obligatorio en Anthropic
        tools=[{"type": "web_search_20250305", "name": "web_search", "max_uses": 3}],
        messages=[{"role": "user", "content": DEVDAY}],
    )
    for bloque in r_claude.content:
        if bloque.type == "text":
            print(bloque.text, end="")
            for c in getattr(bloque, "citations", None) or []:
                print(f"  [{c.url}]", end="")
    print()
    print(resumen_uso("claude-haiku-4-5", r_claude.usage.input_tokens, r_claude.usage.output_tokens))
except Exception as e:
    print(f"⚠️ La búsqueda web de Anthropic no está disponible: {type(e).__name__}: {e}")
    print("Revisa: clave de Anthropic con saldo y 'Web search' habilitado en la Claude Console (Settings → Privacy).")

Basándome en los anuncios del OpenAI DevDay 2025, aquí están los tres más importantes:

• **ChatGPT Apps Platform**: OpenAI introdujo una plataforma de "Apps" que permite a los usuarios interactuar con servicios externos dentro de las conversaciones de ChatGPT, con un directorio de aplicaciones y monetización para desarrolladores  [https://medium.com/@ilanpoonjolai/openais-devday-2025-major-announcements-and-their-implications-b21d09724a8f].

• **AgentKit**: OpenAI anunció AgentKit, un conjunto de herramientas de desarrollador diseñado para simplificar la construcción e implementación de agentes de IA personalizados  [https://medium.com/@ilanpoonjolai/openais-devday-2025-major-announcements-and-their-implications-b21d09724a8f].

• **Sora 2 API**: Los desarrolladores ahora pueden acceder a Sora 2 a través de la API e integrar sus capacidades de generación de video en sus propias aplicaciones  [https://medium.com/@ilanpoonjolai/openais-devday-2025-major-announcements-and-their-implicatio

Tres diferencias con OpenAI:

1. **Sintaxis versionada**: `web_search_20250305`. Anthropic fecha sus herramientas.
2. **Hay que habilitarla** en la Console.
3. **Control explícito** con `max_uses`: cuántas búsquedas puede hacer en una llamada.

Misma funcionalidad, filosofías distintas: Anthropic prioriza control y gobernanza; OpenAI, simplicidad.

## 3.8 · Otras herramientas

- **`file_search`** (OpenAI): RAG gestionado. Subes documentos y OpenAI los trocea y vectoriza. Es exactamente lo
  que construiremos nosotros en el **Módulo 3**, para tener criterio entre usar lo del proveedor o hacerlo propio.
- **`code_interpreter` / `code_execution`**: ejecuta Python en un sandbox. El modelo deja de sumar «de cabeza».
- **`computer_use`**: el modelo maneja un ordenador con capturas, ratón y teclado.
- **MCP remoto**: herramientas propias expuestas al modelo. Lo veremos en el **Módulo 5**, con agentes.

---
# Bloque 4 · Comparativa de proveedores en vivo

## 4.1 · Misma pregunta, distinto proveedor

In [23]:
# 4.1 · El escenario y el system prompt común
COMPARISON_SYSTEM_PROMPT = """Eres un ingeniero de datos de nivel staff que asesora a un equipo de SaaS.

Responde con esta estructura:
1. Recomendación (una frase).
2. Por qué, con números: latencia de las consultas, almacenamiento y coste operativo.
3. Cuándo sería mejor la otra opción.

Restricciones: como máximo 200 palabras. No respondas «depende» sin cuantificar. Nada de marketing de proveedores."""

SCENARIO = """Nuestro SaaS funciona sobre PostgreSQL 16. La tabla de eventos tiene 50 millones de filas y crece 2 millones al mes.
El panel de analítica (agregaciones por cliente y por mes) tarda ahora entre 8 y 12 segundos en cargar.
¿Añadimos un almacén columnar (por ejemplo, ClickHouse) o nos quedamos en Postgres con réplicas de lectura y mejores índices?"""

import time
resultados = {}

In [24]:
# 4.1 · OpenAI (Responses API)
t0 = time.time()
r_oa = openai_client.responses.create(
    model="gpt-4o-mini",
    instructions=COMPARISON_SYSTEM_PROMPT,
    input=SCENARIO,
    temperature=0.2,
)
resultados["openai · gpt-4o-mini"] = ("gpt-4o-mini", r_oa.usage.input_tokens, r_oa.usage.output_tokens,
                                      time.time() - t0, len(r_oa.output_text.split()))
print(r_oa.output_text)
print()
print(resumen_uso("gpt-4o-mini", r_oa.usage.input_tokens, r_oa.usage.output_tokens))

1. Recomendación: Añadir un almacén columnar como ClickHouse.

2. Por qué: Con 50 millones de filas y un crecimiento de 2 millones al mes, las consultas en PostgreSQL pueden volverse lentas debido a la latencia de las consultas, que actualmente es de 8-12 segundos. Un almacén columnar puede reducir esta latencia a menos de 1 segundo en consultas analíticas, gracias a su optimización para lecturas masivas. Además, el almacenamiento en ClickHouse es más eficiente, permitiendo comprimir datos y reducir el coste operativo en almacenamiento a aproximadamente un 30-50% en comparación con PostgreSQL.

3. Cuándo sería mejor la otra opción: Si las consultas son principalmente transaccionales y requieren actualizaciones frecuentes, quedarse en PostgreSQL con réplicas de lectura y mejores índices podría ser más adecuado, ya que evitaría la complejidad de gestionar un sistema adicional. Sin embargo, esto sería viable solo si la latencia se mantiene por debajo de 5 segundos.

[gpt-4o-mini] entrada 

In [25]:
# 4.1 · Anthropic (Messages API)
try:
    if anthropic_client is None:
        raise RuntimeError("No hay ANTHROPIC_API_KEY configurada.")
    t0 = time.time()
    r_an = anthropic_client.messages.create(
        model="claude-haiku-4-5-20251001",
        system=COMPARISON_SYSTEM_PROMPT,           # en Anthropic el system va aparte
        messages=[{"role": "user", "content": SCENARIO}],
        max_tokens=600,                            # obligatorio
        # el SDK actual de Anthropic ya no acepta temperature en messages.create
    )
    texto = r_an.content[0].text
    resultados["anthropic · claude-haiku-4-5"] = ("claude-haiku-4-5", r_an.usage.input_tokens,
                                                  r_an.usage.output_tokens, time.time() - t0, len(texto.split()))
    print(texto)
    print()
    print(resumen_uso("claude-haiku-4-5", r_an.usage.input_tokens, r_an.usage.output_tokens))
except Exception as e:
    print(f"⚠️ Anthropic no disponible: {type(e).__name__}: {e}")

## Recomendación
Implementa ClickHouse como capa analítica separada. PostgreSQL con réplicas solo aplazará el problema.

## Por qué, con números

**PostgreSQL + réplicas (estado actual mejorado):**
- Latencia consultas: 4-6 segundos (mejor, pero insuficiente para UX fluida)
- Almacenamiento: ~200 GB en 12 meses (3 réplicas = 600 GB)
- Coste operativo: ~$800/mes (infraestructura + mantenimiento de 3 nodos)
- Límite: a 200M filas, las agregaciones volverán a degradarse

**ClickHouse (arquitectura OLAP):**
- Latencia consultas: 200-500 ms (40x más rápido)
- Almacenamiento: ~20-30 GB comprimido (1 réplica = 60 GB)
- Coste operativo: ~$300/mes + sincronización
- Escala lineal hasta 1B+ filas sin degradación

**Diferencial a 2 años:** ClickHouse cuesta ~$7.2K, PostgreSQL puro ~$19.2K + degradación de producto.

## Cuándo quedarse en PostgreSQL

Solo si:
- El panel se consulta <5 veces/día (sin presión de UX)
- Los agregados son simples (sumas sin filtros complejos)
- Tu equipo no puede mante

In [26]:
# 4.1 · Lado a lado
print(f"{'Proveedor · modelo':<32}{'entrada':>9}{'salida':>9}{'palabras':>10}{'segundos':>10}{'coste $':>12}")
for etiqueta, (modelo, t_in, t_out, seg, palabras) in resultados.items():
    print(f"{etiqueta:<32}{t_in:>9}{t_out:>9}{palabras:>10}{seg:>10.1f}{calcular_coste(modelo, t_in, t_out):>12.6f}")

Proveedor · modelo                entrada   salida  palabras  segundos     coste $
openai · gpt-4o-mini                  190      213       149       3.3    0.000156
anthropic · claude-haiku-4-5          247      420       170       4.9    0.002347


- ¿Cuál es más útil? ¿Cuál respeta mejor el formato? ¿Alguna dice «depende» sin cuantificar?
- ¿Por qué Anthropic suele contar **más tokens de entrada** para el mismo texto? Cada proveedor tiene su
  tokenizador: los tokens no se pueden comparar entre proveedores.

## 4.2 · Misma pregunta, distinto tier de modelo

In [27]:
# 4.2 · Tiers de OpenAI
openai_models = [
    "gpt-4o-mini",
    "gpt-5.4-mini",   # coméntalo si tu cuenta no tiene acceso
    "gpt-5.4",        # coméntalo si tu cuenta no tiene acceso
]

for modelo in openai_models:
    t0 = time.time()
    r = openai_client.responses.create(model=modelo, instructions=COMPARISON_SYSTEM_PROMPT, input=SCENARIO)
    seg = time.time() - t0
    resultados[f"openai · {modelo}"] = (modelo, r.usage.input_tokens, r.usage.output_tokens, seg,
                                         len(r.output_text.split()))
    print(f"=== {modelo} · {seg:.1f} s · {resumen_uso(modelo, r.usage.input_tokens, r.usage.output_tokens)}")
    print(r.output_text)
    print()

=== gpt-4o-mini · 3.8 s · [gpt-4o-mini] entrada 190 · salida 212 · $0.000156
1. **Recomendación:** Añadir un almacén columnar como ClickHouse.

2. **Por qué:** Las consultas analíticas en ClickHouse pueden ejecutarse a velocidades de milisegundos, en comparación con los 8-12 segundos actuales en PostgreSQL. Un sistema de almacenamiento columnar reduce la latencia de las consultas al filtrar columnas específicas y al reducir el almacenamiento de datos (aproximadamente 10 veces menor para datos similares). Además, ClickHouse es más eficiente en el uso de recursos computacionales, lo que podría disminuir los costes operativos en un 30-50% a largo plazo debido a una reducción en el procesamiento de consultas.

3. **Cuándo sería mejor la otra opción:** Si el volumen de datos no superara los 10 millones de filas y las consultas fueran principalmente transaccionales en lugar de analíticas, entonces optimizar PostgreSQL con índices y réplicas de lectura podría ser suficiente, manteniendo la si

=== gpt-5.4-mini · 4.5 s · [gpt-5.4-mini] entrada 189 · salida 324 · $0.001600
1. **Recomendación:** añade un almacén columnar para analítica y deja PostgreSQL 16 para transaccional.

2. **Por qué, con números:** con 50 M de filas y agregaciones por cliente/mes, Postgres con índices y réplicas suele mejorar el throughput, pero no el coste de escanear y agrupar grandes rangos: bajar de **8–12 s** a **2–5 s** es razonable si la consulta está bien indexada y cacheada, pero no esperes sub-segundo de forma consistente. Un columnar suele llevar esas agregaciones a **200 ms–1.5 s** porque lee 5–20× menos datos efectivos para ese patrón. En almacenamiento, una copia analítica columnar suele ocupar **30–60%** del tamaño lógico de la tabla (más compresión), mientras que una réplica Postgres duplica casi todo: **~100% extra** del almacenamiento y del WAL asociado. En coste operativo, Postgres con réplicas es más simple: **+1–2 horas/semana** de tuning si el patrón no cambia; mantener ambos sistem

=== gpt-5.4 · 14.3 s · [gpt-5.4] entrada 189 · salida 360 · $0.005873
1. **Recomendación:** Quédate en **PostgreSQL** y optimiza con **particionado mensual, índices compuestos/cubrientes y tablas de agregados/materialized views** antes de añadir un almacén columnar.

2. **Por qué, con números:**  
Con **50 M filas** y crecimiento de **2 M/mes**, tu volumen sigue siendo moderado para Postgres 16 si las consultas del panel son “por cliente y por mes”. Un dashboard que hoy tarda **8–12 s** suele bajar a **200–800 ms** si atacas agregaciones repetitivas con preagregados, o a **1–3 s** con buen particionado + índices.  
**Almacenamiento:** 50 M eventos suelen caber en **decenas de GB** en Postgres; añadir ClickHouse casi **duplica** almacenamiento útil (datos + pipelines + staging).  
**Coste operativo:** Postgres optimizado implica **0–1** tecnologías nuevas; ClickHouse añade **ETL/CDC, reconciliación, observabilidad y backfills**, normalmente **0.25–0.5 FTE** extra de operación. Réplicas 

In [28]:
# 4.2 · Tu turno — tiers de Anthropic
anthropic_models = [
    "claude-haiku-4-5-20251001",
    # "claude-sonnet-4-6",
]

for modelo in anthropic_models:
    try:
        if anthropic_client is None:
            raise RuntimeError("No hay ANTHROPIC_API_KEY configurada.")
        r = anthropic_client.messages.create(model=modelo, system=COMPARISON_SYSTEM_PROMPT, max_tokens=600,
                                             messages=[{"role": "user", "content": SCENARIO}])
        print(f"=== {modelo} · {resumen_uso(modelo, r.usage.input_tokens, r.usage.output_tokens)}")
        print(r.content[0].text, "\n")
    except Exception as e:
        print(f"⚠️ {modelo}: {type(e).__name__}: {e}")

=== claude-haiku-4-5-20251001 · [claude-haiku-4-5-20251001] entrada 247 · salida 447 · $0.002482
**Recomendación:** Implementa un almacén columnar (ClickHouse) con sincronización desde PostgreSQL.

**Por qué, con números:**

Con 50M de filas y crecimiento de 2M/mes, tus agregaciones mejoran así:

- **Latencia:** ClickHouse ejecuta esas sumas/conteos en 200-500ms (compresión columnar + paralelización). PostgreSQL con índices optimizados rondaría 3-5s, aún lento para UX moderna.
- **Almacenamiento:** ClickHouse comprime a ~5-10% del volumen original (típico en eventos). 50M filas de eventos ocupan ~200GB en Postgres; en ClickHouse, 10-20GB. Las réplicas en Postgres duplican el coste.
- **Coste operativo:** Una instancia ClickHouse pequeña (8 vCPU, 32GB RAM) cuesta ~$300/mes. Replicar Postgres significa +$200-400/mes. ROI en 2-3 meses si reduces carga de lectura en PostgreSQL.

**Cuándo quedarse en Postgres:**

- Si el panel se consulta <5 veces/día y puedes aceptar 5-8s de latencia.
- Si

In [29]:
# 4.2 · ¿Cuánto más caro es cada tier?
base = calcular_coste("gpt-4o-mini", 1_000_000, 1_000_000)
for modelo in ["gpt-4o-mini", "gpt-5.4-nano", "gpt-5.4-mini", "gpt-5.4", "claude-haiku-4-5", "claude-sonnet-4-6"]:
    p_in, p_out = precio(modelo)
    print(f"{modelo:<20} entrada ${p_in:>5.2f}/M · salida ${p_out:>6.2f}/M · "
          f"{calcular_coste(modelo, 1_000_000, 1_000_000) / base:>5.1f}× gpt-4o-mini")

gpt-4o-mini          entrada $ 0.15/M · salida $  0.60/M ·   1.0× gpt-4o-mini
gpt-5.4-nano         entrada $ 0.20/M · salida $  1.25/M ·   1.9× gpt-4o-mini
gpt-5.4-mini         entrada $ 0.75/M · salida $  4.50/M ·   7.0× gpt-4o-mini
gpt-5.4              entrada $ 2.50/M · salida $ 15.00/M ·  23.3× gpt-4o-mini
claude-haiku-4-5     entrada $ 1.00/M · salida $  5.00/M ·   8.0× gpt-4o-mini
claude-sonnet-4-6    entrada $ 3.00/M · salida $ 15.00/M ·  24.0× gpt-4o-mini


## 4.3 · Criterios de selección

- Un chatbot de soporte que responde FAQs: ¿qué modelo?
- Análisis de contratos legales de 50 páginas: ¿cambia la elección?
- 10.000 usuarios: ¿el mismo modelo para todos, o uno barato por defecto y escalar cuando haga falta?

En producción lo habitual es el **routing / cascada**: un modelo barato para el 80 % de las peticiones y uno
premium solo cuando la tarea lo pide. Lo construimos en la **Sesión 3**.

---
# Bloque 5 · Agregadores: una interfaz, varios proveedores

## 5.2 · LiteLLM en vivo

In [30]:
# 5.2 · El mismo código para los dos proveedores
import litellm
litellm.suppress_debug_info = True

WEBHOOKS_SYSTEM = "Eres un líder de ingeniería backend pragmático. Responde en 150 palabras como máximo, con una recomendación clara."
WEBHOOKS_QUESTION = ("Recibimos 500 webhooks de Stripe por segundo en nuestra API de Rails. ¿Los procesamos de forma síncrona "
                     "dentro de la petición, o los metemos en una cola y los procesan workers en segundo plano?")

for modelo in ["openai/gpt-4o-mini", "anthropic/claude-haiku-4-5-20251001"]:
    try:
        resp = litellm.completion(
            model=modelo,  # ← lo único que cambia
            messages=[
                {"role": "system", "content": WEBHOOKS_SYSTEM},
                {"role": "user", "content": WEBHOOKS_QUESTION},
            ],
            temperature=0.2,
            max_tokens=400,
        )
        print(f"=== {modelo} · entrada {resp.usage.prompt_tokens} · salida {resp.usage.completion_tokens} · "
              f"${litellm.completion_cost(resp):.6f}")
        print(resp.choices[0].message.content, "\n")
    except Exception as e:
        print(f"⚠️ {modelo}: {type(e).__name__}: {str(e)[:160]}\n")

=== openai/gpt-4o-mini · entrada 84 · salida 178 · $0.000119
Te recomiendo encarecidamente que utilices una cola para procesar los webhooks en segundo plano. Procesar los webhooks de forma síncrona puede llevar a tiempos de respuesta prolongados y potencialmente a errores si tu API no puede manejar la carga. Al encolar los eventos, puedes desacoplar la recepción de los webhooks de su procesamiento, lo que mejora la resiliencia y escalabilidad de tu sistema.

Implementa una solución como Sidekiq o Resque para manejar los trabajos en segundo plano. Esto te permitirá procesar los webhooks de manera eficiente, manejar reintentos en caso de fallos y escalar según sea necesario. Además, asegúrate de implementar un mecanismo de confirmación para que Stripe sepa que has recibido y procesado correctamente los webhooks. Esto garantizará la integridad de tus transacciones y la estabilidad de tu API. 



=== anthropic/claude-haiku-4-5-20251001 · entrada 99 · salida 286 · $0.001529
**Recomendación: Cola + workers en segundo plano.**

A 500 webhooks/segundo, procesarlos síncronamente es insostenible:

- **Timeouts**: Stripe espera respuesta en ~5s. Procesos largos fallarán.
- **Bloqueos**: Saturarás threads de Rails, rechazando peticiones legítimas.
- **Sin reintentos**: Si falla, Stripe reintenta pero pierdes control.

**Implementa:**

1. **Recibe webhook** → valida firma Stripe → encola en Redis/SQS
2. **Responde 200 inmediatamente** a Stripe
3. **Workers** (Sidekiq/Resque) procesan en paralelo

**Ventajas:**
- Desacoplamiento
- Reintentos automáticos
- Escalabilidad horizontal
- Mejor observabilidad

**Detalles:**
- Usa `sidekiq` con 50-100 workers
- Implementa idempotencia (Stripe reenvía duplicados)
- Monitorea lag de la cola

Es el estándar en producción. No lo dudes. 



Tres diferencias con las llamadas directas del Bloque 4:

| | Llamada directa (Responses API) | LiteLLM (formato Chat Completions) |
|---|---|---|
| Entrada | `instructions` + `input` | `messages` con roles |
| Texto | `resp.output_text` | `resp.choices[0].message.content` |
| Tokens | `resp.usage.input_tokens` | `resp.usage.prompt_tokens` |

LiteLLM usa **Chat Completions** como idioma común: por eso conviene conocer las dos APIs de OpenAI.

## 5.3 · Cambiar de proveedor en una línea

In [31]:
# 5.3 · evaluate_technical_decision()
DECISION_SYSTEM = """Eres un ingeniero principal. Evalúa la decisión técnica que recibes.
Responde con: Veredicto (ADELANTE / NO SEGUIR / FALTAN DATOS), después las dos razones de más peso y después una cosa que medir primero.
Como máximo 120 palabras."""

def evaluate_technical_decision(decision: str, model: str = "openai/gpt-4o-mini") -> str:
    resp = litellm.completion(
        model=model,
        messages=[{"role": "system", "content": DECISION_SYSTEM}, {"role": "user", "content": decision}],
        temperature=0.2,
        max_tokens=300,
    )
    return resp.choices[0].message.content

DECISION = "Pasar nuestra base de datos Postgres de 3 GB a una base de datos serverless gestionada para ahorrar trabajo de operaciones."

print(evaluate_technical_decision(DECISION))
# print(evaluate_technical_decision(DECISION, model="anthropic/claude-haiku-4-5-20251001"))

Veredicto: ADELANTE

1. La migración a una base de datos serverless puede reducir significativamente la carga operativa, permitiendo que el equipo se enfoque en el desarrollo en lugar de la gestión de infraestructura.
2. Las bases de datos serverless pueden escalar automáticamente según la demanda, lo que puede resultar en un uso más eficiente de los recursos y potencialmente en ahorros de costos.

Cosa a medir primero: Evaluar el patrón de uso de la base de datos actual para entender las necesidades de rendimiento y escalabilidad antes de la migración.


En la **Sesión 3** convertimos esto en la capa de proveedores del proyecto, con **fallback automático**: si OpenAI
se cae, la aplicación pasa a Anthropic sin que el usuario lo note.

---
# Bloque 6 · Tokens: cómo funcionan y cuánto cuestan

## 6.1 · Anatomía de los tokens

In [32]:
# 6.1 · Mismo contenido, distintos formatos
import json
import tiktoken

enc = tiktoken.get_encoding("o200k_base")      # tokenizador de gpt-4o, gpt-4o-mini y la familia gpt-5
enc_viejo = tiktoken.get_encoding("cl100k_base")  # el de gpt-4 y gpt-3.5 (2023)

TEXTOS = {
    "Inglés": "We are migrating our payment service from a monolith to microservices. "
              "Which risks should we consider before we start, and how long will it take?",
    "Español": "Estamos migrando nuestro servicio de pagos de un monolito a microservicios. "
               "¿Qué riesgos deberíamos considerar antes de empezar y cuánto tiempo llevará?",
    "Código Python": "def total(items):\n    return sum(i.price * i.quantity for i in items if i.active)",
    "JSON": json.dumps({"customer_id": 4821, "amount_eur": 129.90, "currency": "EUR", "status": "paid"}),
    "PRODUCTION_SYSTEM_PROMPT": PRODUCTION_SYSTEM_PROMPT,
}

print(f"{'Texto':<26}{'caracteres':>11}{'palabras':>10}{'o200k':>8}{'cl100k':>8}{'car./token':>12}")
for nombre, texto in TEXTOS.items():
    n = len(enc.encode(texto))
    n_viejo = len(enc_viejo.encode(texto))
    print(f"{nombre:<26}{len(texto):>11}{len(texto.split()):>10}{n:>8}{n_viejo:>8}{len(texto) / n:>12.1f}")

es, en = TEXTOS["Español"], TEXTOS["Inglés"]
for etiqueta, e in (("o200k (hoy)", enc), ("cl100k (2023)", enc_viejo)):
    print(f"\nEspañol frente a inglés con {etiqueta}: {len(e.encode(es)) / len(e.encode(en)) - 1:+.0%} tokens")

Texto                      caracteres  palabras   o200k  cl100k  car./token


Inglés                            145        25      30      30         4.8
Español                           152        22      31      40         4.9
Código Python                      81        12      20      20         4.0
JSON                               79         8      30      30         2.6
PRODUCTION_SYSTEM_PROMPT          953       156     229     257         4.2

Español frente a inglés con o200k (hoy): +3% tokens

Español frente a inglés con cl100k (2023): +33% tokens


In [33]:
# 6.1 · Token a token
FRASE = "¿Cuánto cuesta migrar una base de datos PostgreSQL a Amazon Aurora?"

ids = enc.encode(FRASE)
piezas = [enc.decode_single_token_bytes(t).decode("utf-8", errors="replace") for t in ids]
print(f"{len(ids)} tokens:")
print(" | ".join(piezas))

15 tokens:
¿Cu | ánto |  cuesta |  migr | ar |  una |  base |  de |  datos |  Postgre | SQL |  a |  Amazon |  Aurora | ?


In [34]:
# 6.1 · Tu turno: tokeniza tu texto o tu system prompt del Bloque 2
MI_TEXTO = "Escribe aquí tu texto"

ids = enc.encode(MI_TEXTO)
print(len(ids), "tokens →", " | ".join(enc.decode_single_token_bytes(t).decode("utf-8", errors="replace") for t in ids))

5 tokens → Es | cribe |  aquí |  tu |  texto


## 6.2 · Entrada vs salida: la asimetría que define tu factura

In [35]:
# 6.2 · Misma pregunta, tres longitudes de respuesta
medidas = []
for palabras in (50, 200, 500):
    r = openai_client.responses.create(
        model=MODEL,
        instructions=PRODUCTION_SYSTEM_PROMPT.replace("como máximo 250 palabras", f"unas {palabras} palabras"),
        input=USER_QUESTION,
    )
    p_in, p_out = precio(MODEL)
    coste_in = r.usage.input_tokens * p_in / 1e6
    coste_out = r.usage.output_tokens * p_out / 1e6
    medidas.append((r.usage.input_tokens, r.usage.output_tokens))
    print(f"~{palabras:>3} palabras · entrada {r.usage.input_tokens:>4} tok (${coste_in:.6f}) · "
          f"salida {r.usage.output_tokens:>4} tok (${coste_out:.6f}) · la salida es el "
          f"{coste_out / (coste_in + coste_out):.0%} del coste")

~ 50 palabras · entrada  264 tok ($0.000040) · salida  103 tok ($0.000062) · la salida es el 61% del coste


~200 palabras · entrada  264 tok ($0.000040) · salida  274 tok ($0.000164) · la salida es el 81% del coste


~500 palabras · entrada  264 tok ($0.000040) · salida  573 tok ($0.000344) · la salida es el 90% del coste


In [36]:
# 6.2 · Proyección a escala: 10.000 usuarios × 5 llamadas/día × 30 días
LLAMADAS_MES = 10_000 * 5 * 30
t_in, t_out = medidas[1]  # la respuesta mediana de la celda anterior

print(f"{LLAMADAS_MES:,} llamadas/mes con {t_in} tokens de entrada y {t_out} de salida por llamada\n")
filas = []
for modelo in ["gpt-4o-mini", "gpt-5.4-nano", "gpt-5.4-mini", "claude-haiku-4-5", "gpt-5.4", "claude-sonnet-4-6"]:
    filas.append((modelo, calcular_coste(modelo, t_in, t_out) * LLAMADAS_MES))
barato = filas[0][1]
for modelo, mes in filas:
    print(f"{modelo:<20} ${mes:>10,.0f}/mes   {mes / barato:>5.1f}×")

1,500,000 llamadas/mes con 264 tokens de entrada y 274 de salida por llamada

gpt-4o-mini          $       306/mes     1.0×
gpt-5.4-nano         $       593/mes     1.9×
gpt-5.4-mini         $     2,146/mes     7.0×
claude-haiku-4-5     $     2,451/mes     8.0×
gpt-5.4              $     7,155/mes    23.4×
claude-sonnet-4-6    $     7,353/mes    24.0×


## 6.3 · El coste oculto de las conversaciones multi-turno

Cada llamada **reenvía todo el historial**.

In [37]:
# 6.3 · Respuesta a un incidente en 5 turnos
INCIDENT_SYSTEM = "Eres un SRE que ayuda durante un incidente en producción. Sé conciso: como máximo 120 palabras por respuesta."
TURNOS = [
    "La latencia p99 de nuestra API de checkout ha pasado de 200 ms a 3 s hace diez minutos. ¿Por dónde empezamos?",
    "La CPU está normal, pero el pool de conexiones de la base de datos está al 100 % de uso.",
    "Hace 15 minutos desplegamos una versión nueva que añade una consulta para cargar los puntos de fidelización del cliente.",
    "La consulta nueva no tiene índice en customer_id y la tabla tiene 40 millones de filas. ¿Rollback o hotfix?",
    "Hemos hecho rollback y la latencia se está recuperando. ¿Qué debería ir en el postmortem?",
]

historial = []
filas = []
acumulado = 0.0
for n, mensaje in enumerate(TURNOS, start=1):
    historial.append({"role": "user", "content": mensaje})
    r = openai_client.responses.create(model=MODEL, instructions=INCIDENT_SYSTEM, input=historial)
    historial.append({"role": "assistant", "content": r.output_text})
    coste = calcular_coste(MODEL, r.usage.input_tokens, r.usage.output_tokens)
    acumulado += coste
    filas.append((n, r.usage.input_tokens, r.usage.output_tokens, coste, acumulado))

print(f"{'Turno':>5}{'entrada':>9}{'salida':>8}{'coste turno $':>15}{'acumulado $':>14}")
for n, t_in, t_out, coste, acum in filas:
    print(f"{n:>5}{t_in:>9}{t_out:>8}{coste:>15.6f}{acum:>14.6f}")
print(f"\nEl turno 5 envía {filas[-1][1] / filas[0][1]:.1f}× más tokens de entrada que el turno 1.")

Turno  entrada  salida  coste turno $   acumulado $
    1       66     121       0.000082      0.000082
    2      217     171       0.000135      0.000218
    3      419     157       0.000157      0.000375
    4      608     132       0.000170      0.000545
    5      770     171       0.000218      0.000763

El turno 5 envía 11.7× más tokens de entrada que el turno 1.


In [38]:
# 6.3 · El system prompt se paga en cada turno
tokens_prompt = len(enc.encode(PRODUCTION_SYSTEM_PROMPT))
for turnos in (1, 5, 20):
    total = tokens_prompt * turnos
    print(f"{turnos:>2} turnos → {total:>6,} tokens de entrada solo en system prompt "
          f"(${calcular_coste(MODEL, total, 0):.6f} con {MODEL}, ${calcular_coste('gpt-5.4', total, 0):.6f} con gpt-5.4)")

 1 turnos →    229 tokens de entrada solo en system prompt ($0.000034 con gpt-4o-mini, $0.000572 con gpt-5.4)
 5 turnos →  1,145 tokens de entrada solo en system prompt ($0.000172 con gpt-4o-mini, $0.002863 con gpt-5.4)
20 turnos →  4,580 tokens de entrada solo en system prompt ($0.000687 con gpt-4o-mini, $0.011450 con gpt-5.4)


Cómo se mitiga (lo trabajamos en sesiones posteriores):

- Truncar el historial (ventana deslizante).
- Resumir los turnos antiguos.
- *Prompt caching*: Anthropic cobra los aciertos de caché al 10 % del precio de entrada.
- `previous_response_id` en la Responses API: el historial se queda en el servidor de OpenAI.

En un producto con conversaciones largas, gestionar el contexto **no es una optimización: es viabilidad económica**.

---
# Bloque 7 · Arquitecturas reales y cierre

Para cada producto: ¿cómo abstrae el prompting? ¿Qué modelo usa probablemente? ¿Cómo gestiona los tokens?
¿Hace routing entre modelos?

| Producto | Abstracción | Modelo | Tokens | Routing |
|---|---|---|---|---|
| GitHub Copilot | Prompt construido con el fichero, imports y comentarios | Pequeño para autocompletar, grande para el chat | Fichero actual y vecinos, no el repo | Sí |
| Soporte tipo Intercom / Zendesk AI | Políticas + datos del cliente + historial de tickets | Gama media | Prompt largo: el caching es clave | A veces |
| Análisis de documentos tipo NotebookLM | El usuario sube documentos y pregunta | Ventana de contexto grande | 50 páginas = 50.000+ tokens por pregunta | Sí → **RAG, Módulo 3** |

## Extra · Las tres interfaces lado a lado

In [39]:
# Extra · Misma pregunta en las tres interfaces
SYSTEM = "Eres un asesor técnico conciso. Responde en 60 palabras como máximo."
QUESTION = "¿Cuándo debería una startup pasar de un monolito a microservicios?"

# 1 · Responses API (OpenAI directo)
r1 = openai_client.responses.create(model="gpt-4o-mini", instructions=SYSTEM, input=QUESTION)
print("1 · Responses API ·", r1.usage.input_tokens, "→", r1.usage.output_tokens, "tokens\n", r1.output_text, "\n")

# 2 · Messages API (Anthropic)
try:
    if anthropic_client is None:
        raise RuntimeError("No hay ANTHROPIC_API_KEY configurada.")
    r2 = anthropic_client.messages.create(model="claude-haiku-4-5-20251001", system=SYSTEM, max_tokens=200,
                                          messages=[{"role": "user", "content": QUESTION}])
    print("2 · Messages API ·", r2.usage.input_tokens, "→", r2.usage.output_tokens, "tokens\n", r2.content[0].text, "\n")
except Exception as e:
    print(f"2 · Messages API · ⚠️ {type(e).__name__}: {e}\n")

# 3 · LiteLLM (formato Chat Completions)
r3 = litellm.completion(model="openai/gpt-4o-mini",
                        messages=[{"role": "system", "content": SYSTEM}, {"role": "user", "content": QUESTION}])
print("3 · LiteLLM ·", r3.usage.prompt_tokens, "→", r3.usage.completion_tokens, "tokens\n",
      r3.choices[0].message.content)

1 · Responses API · 44 → 80 tokens
 Una startup debería considerar pasar de un monolito a microservicios cuando: 

1. El crecimiento del producto demanda escalabilidad.
2. Las funcionalidades se vuelven complejas y difíciles de manejar.
3. Se necesita un despliegue independiente de componentes.
4. Hay requerimientos de diferentes tecnologías para diversas partes del sistema.

Evaluar costos y beneficios es crucial antes de la transición. 



2 · Messages API · 56 → 143 tokens
 Cuando tengas:

1. **Equipos múltiples** trabajando en paralelo (cuellos de botella en deploy)
2. **Escalado selectivo** necesario (solo ciertos componentes requieren más recursos)
3. **Tecnologías diversas** requeridas para diferentes funciones
4. **Problemas de rendimiento** claros en el monolito

**Evita migrar** si aún tienes < 50 desarrolladores o si las ganancias no justifican la complejidad operativa. Los microservicios traen deuda técnica significativa. 



3 · LiteLLM · 44 → 85 tokens
 Una startup debería considerar pasar de un monolito a microservicios cuando: 1) el crecimiento del producto genera complejidad y dificultades en el desarrollo; 2) se requiere escalabilidad específica; 3) hay múltiples equipos trabajando en diferentes funcionalidades; y 4) se necesita un despliegue más ágil y frecuente. Evalúa la madurez del producto y la capacidad del equipo antes de decidir.


## Cierre

1. La calidad de un producto con IA la diseña quien desarrolla, no el usuario. El system prompt es vuestra
   herramienta principal.
2. No existe el mejor modelo: existe el mejor modelo para vuestro caso de uso, presupuesto y volumen.
3. Los tokens son la moneda de este mundo. Controlarlos es controlar vuestro margen.

**Próxima sesión (2):** nace el proyecto del máster, un estimador de proyectos con FastAPI, y la arquitectura CAG.